> 在线讲义：[第 39 章 · 归一化与信号传播：LayerNorm / RMSNorm 的几何、不变性、梯度，Pre-LN 与 Post-LN](https://wikieden.github.io/math-and-eval-stats/lessons/39-normalization.html) · 本书《数学与实验统计》

# 第 39 章 · 归一化与信号传播：LayerNorm / RMSNorm 的几何、不变性、梯度，Pre-LN 与 Post-LN

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/39-normalization.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 几何 | 减均值、除 σ | `layernorm(X)`、`rmsnorm(X)` | 长度都是 2 |
| 第二步 不变性 | 平移、缩放 | `layernorm(3 * X + 5)` | 不变 |
| 第三步 梯度 | 投影 | `ln_grad(X, g)` | Σ = 0、⊥ y |
| 第四步 残差流 | Pre / Post | `residual_variance(12, mode)` | 12.84 vs 1.0 |

例子：向量 x = [2, 4, 6, 8]；LayerNorm / RMSNorm 的几何和不变性；LayerNorm 的梯度方向；
Pre-LN 残差流的方差随层数增长、Post-LN 每层被拉回 1。
里程碑 ME-M24 的“比较 Pre-LN / Post-LN 12 层前向的梯度范数”留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
X = np.array([2.0, 4.0, 6.0, 8.0])

**函数 `layernorm`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def layernorm(x: np.ndarray, eps: float = 0.0) -> np.ndarray:
    m = x.mean(-1, keepdims=True)
    return (x - m) / np.sqrt(((x - m) ** 2).mean(-1, keepdims=True) + eps)

**函数 `rmsnorm`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def rmsnorm(x: np.ndarray, eps: float = 0.0) -> np.ndarray:
    return x / np.sqrt((x ** 2).mean(-1, keepdims=True) + eps)

**函数 `ln_grad`**：y = LN(x)（无仿射）：∂L/∂x = (g − mean(g) − y·mean(g·y)) / σ —— 和第 36 章 BatchNorm 同一个式子，只是沿特征维。

In [ ]:
def ln_grad(x: np.ndarray, g: np.ndarray) -> np.ndarray:
    """y = LN(x)（无仿射）：∂L/∂x = (g − mean(g) − y·mean(g·y)) / σ —— 和第 36 章 BatchNorm 同一个式子，只是沿特征维。"""
    y = layernorm(x)
    sigma = np.sqrt(((x - x.mean()) ** 2).mean())
    return (g - g.mean() - y * (g * y).mean()) / sigma

**函数 `residual_variance`**：残差流 x（初始每个分量方差 1），每层子网络 F 是随机线性层（输出方差 ≈ 输入方差）。

In [ ]:
def residual_variance(layers: int, mode: str, d: int = 512, n: int = 2000, seed: int = 0) -> list[float]:
    """残差流 x（初始每个分量方差 1），每层子网络 F 是随机线性层（输出方差 ≈ 输入方差）。
    Pre-LN：x ← x + F(LN(x))；Post-LN：x ← LN(x + F(x))。返回每层后残差流的方差。"""
    rng = np.random.default_rng(seed)
    x = rng.standard_normal((n, d))
    out = []
    for _ in range(layers):
        w = rng.normal(0, 1 / np.sqrt(d), (d, d))
        x = x + layernorm(x) @ w if mode == "pre" else layernorm(x + x @ w)
        out.append(float(x.var()))
    return out

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 LayerNorm 的几何：减均值 = 投影，除标准差 = 拉到半径 √d 的球面")
y = layernorm(X)
print(f"  x = {X.tolist()} → LN(x) = {np.round(y, 4).tolist()}；分量和 {y.sum():.1f}，长度 {np.linalg.norm(y):.4f}（√4 = 2）")
r = rmsnorm(X)
print(f"  RMSNorm(x) = {np.round(r, 4).tolist()}；不减均值，长度也是 {np.linalg.norm(r):.4f}")

In [ ]:
print("\n第二步 不变性")
print(f"  LN(3x + 5) = {np.round(layernorm(3 * X + 5), 4).tolist()}（和 LN(x) 一样：平移、缩放都不变）")
print(f"  RMSNorm(3x) = {np.round(rmsnorm(3 * X), 4).tolist()}（缩放不变）；RMSNorm(x + 5) = {np.round(rmsnorm(X + 5), 4).tolist()}（平移会变）")

In [ ]:
print("\n第三步 LayerNorm 的梯度：去掉“全 1”方向和 y 方向")
g = np.array([1.0, -2.0, 0.5, 3.0])
dx = ln_grad(X, g)
print(f"  上游梯度 {g.tolist()} → ∂L/∂x = {np.round(dx, 4).tolist()}；Σ = {dx.sum() + 0:.1e}，和 y 的内积 {dx @ layernorm(X) + 0:.1e}")

In [ ]:
print("\n第四步 残差流的方差：Pre-LN 一路累加，Post-LN 每层拉回 1")
pre, post = residual_variance(12, "pre"), residual_variance(12, "post")
print(f"  Pre-LN 第 1、4、8、12 层后：{[round(pre[i], 2) for i in (0, 3, 7, 11)]}（约 1 + L）")
print(f"  Post-LN 第 1、4、8、12 层后：{[round(post[i], 2) for i in (0, 3, 7, 11)]}")

## 3. 代码里才看得到的

1. **eps 默认取 0**：为了让“长度正好 √d”“梯度和精确正交”这些性质在数值上成立；真实实现里 eps ≈ 1e-5，防止 σ = 0。
2. **残差分支用 1/√d 的随机矩阵**：保证 F 的输出方差约等于输入方差（第 36 章的初始化），所以 Pre-LN 每层正好加 1。
3. **没有算梯度范数**：“Pre-LN / Post-LN 12 层的梯度范数对比”留给自己做（里程碑 ME-M24）。

## 4. 自己改一改

1. 在 Post-LN 版本里把 `layernorm` 换成 `rmsnorm`，方差还是 1 吗？
2. 给 Pre-LN 的残差分支乘 1/√(2L)（GPT-2 的做法），看 12 层后的方差。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> LayerNorm 先投影再拉到半径根号 d；
> LN 平移缩放都不变，RMSNorm 只管缩放、少一步更快；
> 梯度投影掉看不见的方向，权重越大步子越小；
> Post-LN 每层拉回、梯度大要预热；
> Pre-LN 残差干净、方差一加加到 1 加 L。